# Lab 09 · Delta Lake

**Part A** (about 50 min): create a Delta table, read its transaction log, update, delete, merge, travel in time, optimize, vacuum and read the change feed.<br>
**Part B** (about 70 min): 11 problems in Spark SQL: GROUP BY, CASE, HAVING, CTEs, window functions, and Delta tables in SQL.

**Checklist questions Part A gives you evidence for:**<br>
<span style="color:#d93025"><b>7.1</b> The transaction log · <b>7.2</b> ACID in Delta Lake · <b>7.3</b> Schema enforcement · <b>7.4</b> How OPTIMIZE works · <b>7.5</b> Z-ordering · <b>7.6</b> UPDATE and DELETE on Delta · <b>7.7</b> MERGE INTO (upsert) · <b>7.8</b> Snapshot isolation · <b>7.9</b> Why VACUUM is necessary · <b>7.10</b> Change Data Feed</span>

## How to work
* Run the **Setup** cell first, then answer the questions in order. `F` is `pyspark.sql.functions`; `DeltaTable` is imported.
* **Part A**: run the code under the question (or add a line), look at the output, and write your answer as a comment. Every Delta table lives under `data/shop/`, which the setup cell clears.
* **Part B**: write each answer as one `spark.sql(...)` query (B9 to B11 need a few statements). Your result must look exactly like the **expected output**.
* The answers are collapsed at the very end of the notebook. Open one only after you have tried.

## Documentation
* [Delta Lake quickstart](https://docs.delta.io/latest/quick-start.html)
* [Table batch reads and writes](https://docs.delta.io/latest/delta-batch.html): schema enforcement, `mergeSchema`, time travel
* [Update, delete and merge](https://docs.delta.io/latest/delta-update.html)
* [Optimizations](https://docs.delta.io/latest/optimizations-oss.html): `OPTIMIZE`, Z-ordering, data skipping
* [Utility commands](https://docs.delta.io/latest/delta-utility.html): `VACUUM`, `history`, `detail`
* [Change data feed](https://docs.delta.io/latest/delta-change-data-feed.html)
* [The Delta transaction log protocol](https://github.com/delta-io/delta/blob/master/PROTOCOL.md)
* [Spark SQL: CTE](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-cte.html), [window functions](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-window.html), [CASE](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-case.html), [EXISTS](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-subqueries.html)

## The data
Three small tables of an online shop. The setup cell creates them, and you can read every row in it.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date* (the day the customer registered)
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)

Good to know:
* order 1011 is a guest checkout: it has no *customer_id*
* customer 6 has no email and no city
* customer 7 has never ordered, and product 107 has never been sold
* order 1016 is for product 108, which is no longer in *products*

For this lab the setup cell also registers the three tables as SQL views (`customers`, `products`, `orders`) and a small view **order_updates** with two changed or new orders (for B10).

## Setup
Run this cell once. It starts Spark with Delta Lake, saves the three shop tables as Parquet files in `data/shop/`, and registers them as SQL views. The first start may print a list of packages: that is Spark loading the Delta library.

In [ ]:
# Setup: starts Spark and creates the shop tables. Run it once.
import logging, os, shutil, sys, time
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, functions as F
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

builder = (SparkSession.builder
           .master("local[4]")   # 4 tasks at a time on your machine
           .appName("lab09")
           .config("spark.sql.adaptive.enabled", "true")        # AQE on, as in Spark's default
           .config("spark.sql.session.timeZone", "UTC")
           .config("spark.sql.shuffle.partitions", "4")          # small data: 4 are enough
           .config("spark.sql.warehouse.dir", os.path.abspath("data/shop/warehouse"))   # SQL tables of Part B
           .config("spark.ui.showConsoleProgress", "false")
           .config("spark.log.level", "FATAL")                  # hide Spark's log messages
           # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
           .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
spark = configure_spark_with_delta_pip(builder).getOrCreate()
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers_rows = [
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
]
products_rows = [
    # product_id, product_name, category, price
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
]
orders_rows = [
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
]

# Write the tables as Parquet files and read them back, the way real data arrives.
# data/shop belongs to this lab: it is rebuilt every time you run this cell.
shutil.rmtree("data/shop", ignore_errors=True)
for table, rows, schema in [
    ("customers", customers_rows, "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE"),
    ("products",  products_rows,  "product_id INT, product_name STRING, category STRING, price DOUBLE"),
    ("orders",    orders_rows,    "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING"),
]:
    spark.createDataFrame(rows, schema).coalesce(1).write.parquet(f"data/shop/{table}")

customers = spark.read.parquet("data/shop/customers")
products = spark.read.parquet("data/shop/products")
orders = spark.read.parquet("data/shop/orders")

# The same tables as SQL views, for Part B
customers.createOrReplaceTempView("customers")
products.createOrReplaceTempView("products")
orders.createOrReplaceTempView("orders")
spark.createDataFrame([
    (1001, 1, 101, 2, date(2024, 1, 10), "delivered"),     # order 1001 with a new quantity
    (1017, 7, 107, 1, date(2024, 3, 30), "shipped"),       # a new order
], orders.schema).createOrReplaceTempView("order_updates")

print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

Look at the tables:

In [ ]:
customers.show()
products.show()
orders.show()
spark.table("order_updates").show()

## Part A · A Delta table, step by step

### The transaction log <span style="color:#d93025">(7.1, 7.2, 7.8)</span>

**A1. Write `orders` as a Delta table to `data/shop/delta/orders`. Which files are in `_delta_log`, and which actions does the first commit file contain (one JSON action per line)?**

In [ ]:
# My answer:
orders.write.format("delta").mode("overwrite").save("data/shop/delta/orders")
print(sorted(os.listdir("data/shop/delta/orders/_delta_log")))

**A2. Run an UPDATE and a DELETE (cell below). In the table's history, how many files did each operation remove and add?**

In [ ]:
# My answer:
table = DeltaTable.forPath(spark, "data/shop/delta/orders")
table.update("status = 'shipped'", {"status": F.lit("delivered")})      # every shipped order is delivered now
table.delete("status = 'cancelled'")

**A3. Append three orders that have an extra column, `discount`. What happens? Then append them again with `option("mergeSchema", "true")`.**

In [ ]:
# My answer:
with_discount = spark.createDataFrame([
    (2001, 1, 101, 1, date(2024, 4, 1), "delivered", 0.1),
    (2002, 2, 105, 2, date(2024, 4, 2), "shipped",   0.0),
    (2003, 4, 106, 1, date(2024, 4, 3), "delivered", 0.2),
], "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING, discount DOUBLE")
with_discount.write.format("delta").mode("append").save("data/shop/delta/orders")

**A4. Read version 0 of the table and the current version. How many rows does each have?**

In [ ]:
# My answer:
version_0 = spark.read.format("delta").option("versionAsOf", 0).load("data/shop/delta/orders")

### MERGE <span style="color:#d93025">(7.7)</span>

**A5. Upsert `changes` into the table with MERGE: update the orders that exist (only if something changed) and insert the new ones. How many rows were updated and inserted?**

In [ ]:
# My answer:
table = DeltaTable.forPath(spark, "data/shop/delta/orders")       # load it again: the schema may have changed
changes = spark.createDataFrame([
    (1001, 1, 101, 2, date(2024, 1, 10), "delivered"),    # quantity changed from 1 to 2
    (1002, 1, 105, 3, date(2024, 1, 10), "delivered"),    # nothing changed
    (1017, 7, 107, 1, date(2024, 3, 30), "shipped"),      # a new order
], orders.schema)

### Maintenance: OPTIMIZE, Z-order and VACUUM <span style="color:#d93025">(7.4, 7.5, 7.9)</span>

**A6. Append 30 small batches to a new table (30 small files). How many files does a filter on one customer read? Run `OPTIMIZE ... ZORDER BY customer_id` and count again.**

In [ ]:
# My answer:
shutil.rmtree("data/shop/delta/many", ignore_errors=True)
for i in range(30):                                                  # 30 small appends = 30 small files
    (spark.range(i * 10_000, (i + 1) * 10_000)
          .select(F.col("id").alias("order_id"),
                  (F.pmod(F.xxhash64("id"), F.lit(5000)) + 1).alias("customer_id"),
                  F.pmod(F.xxhash64("id", F.lit(2)), F.lit(1000)).cast("double").alias("amount"))
          .coalesce(1).write.format("delta").mode("append").save("data/shop/delta/many"))
many = DeltaTable.forPath(spark, "data/shop/delta/many")

def files_read(customer_id):
    """How many data files a filter on one customer reads."""
    return (spark.read.format("delta").load("data/shop/delta/many")
                 .filter(F.col("customer_id") == customer_id)
                 .select("_metadata.file_path").distinct().count())

print("files:", many.detail().first()["numFiles"], "| read for customer 42:", files_read(42))

**A7. Run `VACUUM` with 0 hours of retention (the safety check is turned off for this practice table), then try to read version 0 of the table. What happens?**

In [ ]:
# My answer:
import glob
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")   # allow 0 hours, for practice only
print(len(glob.glob("data/shop/delta/many/*.parquet")), "Parquet files on disk before")
many.vacuum(0)
print(len(glob.glob("data/shop/delta/many/*.parquet")), "after")

### Change Data Feed <span style="color:#d93025">(7.10)</span>

**A8. Create a table with the change data feed turned on, update one order and delete another (cell below). What does the change feed show since version 1?**

In [ ]:
# My answer:
orders.write.format("delta").mode("overwrite").option("delta.enableChangeDataFeed", "true").save("data/shop/delta/cdf")
cdf_table = DeltaTable.forPath(spark, "data/shop/delta/cdf")
cdf_table.update("order_id = 1001", {"status": F.lit("returned")})
cdf_table.delete("order_id = 1006")

**Done with Part A?** Explain each red checklist question at the top aloud, in your own words, using what you saw.

## Part B · Spark SQL
11 problems. Answer each one with `spark.sql(""" ... """).show()` on the views `customers`, `products`, `orders` (and `order_updates` in B10).

**B1. How many orders and how many items (sum of quantity) are there for each status? Columns: status, orders, items; sorted by status.**

Expected output:

```
+---------+------+-----+
|   status|orders|items|
+---------+------+-----+
|cancelled|     2|    3|
|delivered|    10|   20|
|  shipped|     4|    6|
+---------+------+-----+
```

<details><summary>Hint</summary>

`SELECT status, COUNT(*) AS orders, SUM(quantity) AS items FROM orders GROUP BY status ORDER BY status`

</details>

**B2. Give every order a size with CASE: `small` (quantity 1), `medium` (2 to 3) or `large` (4 or more), and count the orders of each size. Columns: size, orders; sorted by size.**

Expected output:

```
+------+------+
|  size|orders|
+------+------+
| large|     2|
|medium|     5|
| small|     9|
+------+------+
```

<details><summary>Hint</summary>

`CASE WHEN ... THEN ... ELSE ... END AS size`; `GROUP BY 1` groups by the first selected column.

</details>

**B3. Which customers spent more than 100 on delivered orders (quantity × price)? Columns: name, spent; highest first.**

Expected output:

```
+-------------+------+
|         name| spent|
+-------------+------+
|Bella Schmidt|1525.0|
| Aziz Karimov|1395.0|
+-------------+------+
```

<details><summary>Hint</summary>

`WHERE` filters rows before grouping, `HAVING` filters groups after it.

</details>

**B4. With a CTE and a window function: the best-selling product (most items delivered) of each category. Columns: category, product_name, items.**

Expected output:

```
+-----------+------------+-----+
|   category|product_name|items|
+-----------+------------+-----+
|      Books|       Novel|   12|
|Electronics|  Headphones|    3|
|    Kitchen|     Blender|    1|
+-----------+------------+-----+
```

<details><summary>Hint</summary>

`WITH items AS (...), ranked AS (...) SELECT ...`; `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)`.

</details>

**B5. For customers 1 and 2, show each order with the customer's running total of items by date. Columns: customer_id, order_id, order_date, quantity, items_so_far.**

Expected output:

```
+-----------+--------+----------+--------+------------+
|customer_id|order_id|order_date|quantity|items_so_far|
+-----------+--------+----------+--------+------------+
|          1|    1001|2024-01-10|       1|           1|
|          1|    1002|2024-01-10|       3|           4|
|          1|    1007|2024-02-20|       1|           5|
|          1|    1015|2024-03-25|       1|           6|
|          2|    1003|2024-01-25|       2|           2|
|          2|    1005|2024-02-05|       1|           3|
|          2|    1013|2024-03-18|       1|           4|
+-----------+--------+----------+--------+------------+
```

<details><summary>Hint</summary>

`SUM(quantity) OVER (PARTITION BY ... ORDER BY ... ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`

</details>

**B6. Which customers have no orders? Use `NOT EXISTS`. Columns: customer_id, name.**

Expected output:

```
+-----------+---------+
|customer_id|     name|
+-----------+---------+
|          7|Grace Lee|
+-----------+---------+
```

<details><summary>Hint</summary>

`WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)`

</details>

**B7. Delivered revenue per month and its share of the total, in percent (1 decimal). Columns: month, revenue, share_pct.**

Expected output:

```
+-----+-------+---------+
|month|revenue|share_pct|
+-----+-------+---------+
|    1| 1545.0|     49.6|
|    2|  175.0|      5.6|
|    3| 1395.0|     44.8|
+-----+-------+---------+
```

<details><summary>Hint</summary>

`SUM(revenue) OVER ()` is the total over all rows.

</details>

**B8. For each customer, the first and the last order date and the days between them. Columns: name, first_order, last_order, days_between; sorted by name.**

Expected output:

```
+----------------+-----------+----------+------------+
|            name|first_order|last_order|days_between|
+----------------+-----------+----------+------------+
|    Aziz Karimov| 2024-01-10|2024-03-25|          75|
|   Bella Schmidt| 2024-01-25|2024-03-18|          53|
|     Carlos Ruiz| 2024-02-03|2024-03-28|          54|
|Dilnoza Rahimova| 2024-02-15|2024-03-08|          22|
|     Emma Wilson| 2024-03-02|2024-03-20|          18|
|    Farid Aliyev| 2024-03-12|2024-03-12|           0|
+----------------+-----------+----------+------------+
```

<details><summary>Hint</summary>

`MIN`, `MAX` and `DATEDIFF(end, start)`.

</details>

**B9. Create the Delta table `shop_orders` from orders with `CREATE TABLE ... AS SELECT`, mark every shipped order as delivered with an SQL `UPDATE`, then count the orders per status.**

Expected output:

```
+---------+------+
|   status|orders|
+---------+------+
|cancelled|     2|
|delivered|    14|
+---------+------+
```

<details><summary>Hint</summary>

`CREATE TABLE shop_orders USING DELTA AS SELECT * FROM orders`, then `UPDATE shop_orders SET ... WHERE ...`.

</details>

**B10. MERGE the view `order_updates` into `shop_orders`: update the matching orders, insert the new ones. Then show order_id, quantity and status of orders 1001 and 1017.**

Expected output:

```
+--------+--------+---------+
|order_id|quantity|   status|
+--------+--------+---------+
|    1001|       2|delivered|
|    1017|       1|  shipped|
+--------+--------+---------+
```

<details><summary>Hint</summary>

`MERGE INTO target t USING source s ON ... WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *`

</details>

**B11. Count the rows of `shop_orders` in versions 0, 1 and 2 with `VERSION AS OF`. Columns: version, rows.**

Expected output:

```
+-------+----+
|version|rows|
+-------+----+
|      0|  16|
|      1|  16|
|      2|  17|
+-------+----+
```

<details><summary>Hint</summary>

`SELECT COUNT(*) FROM shop_orders VERSION AS OF 0`, three times, glued with `UNION ALL`.

</details>

---
**Done?** In the Theory part of Day 9, answer the red checklist questions aloud.

## Answers
Try each question yourself first. Click a question to open its answer; Part A answers also show the output of the reference run.

<details>
<summary><b>A1</b> &nbsp;Write orders as a Delta table to data/shop/delta/orders. Which files are in _delta_log, and which actions ...</summary>

**One JSON file per commit (`00000000000000000000.json`, plus checksum files), and the actions `commitInfo`, `metaData`, `protocol` and one `add` per data file.** This **transaction log** is the table's source of truth: a reader lists the log, replays the actions in order, and gets the exact set of Parquet files that make up a version. A commit counts only once its JSON file exists, and two writers can't create the same file, so a write is all or nothing: **ACID** (atomic, consistent, isolated, durable). Every query reads one fixed version (**snapshot isolation**): writers add new files and a new commit, and never change the files a reader is using, so a long query never sees half of someone else's write.

```python
orders.write.format("delta").mode("overwrite").save("data/shop/delta/orders")
print(sorted(os.listdir("data/shop/delta/orders/_delta_log")))
import json
with open("data/shop/delta/orders/_delta_log/00000000000000000000.json") as commit:
    print([list(json.loads(line))[0] for line in commit])
```

Result:

```
['.00000000000000000000.crc.crc', '.00000000000000000000.json.crc', '00000000000000000000.crc', '00000000000000000000.json', '_staged_commits']
['commitInfo', 'metaData', 'protocol', 'add']
```

</details>

<details>
<summary><b>A2</b> &nbsp;Run an UPDATE and a DELETE (cell below). In the table's history, how many files did each operation remove and ...</summary>

**Each removed 1 file and added 1.** Yes, Delta supports `UPDATE`, `DELETE` and `MERGE`, in SQL and in Python. Parquet files can't change (Lab 04), so Delta rewrites only the files that contain affected rows: it writes new files and commits `remove` actions for the old ones and `add` actions for the new ones. The old files stay on disk for time travel until `VACUUM` (A7).

```python
table = DeltaTable.forPath(spark, "data/shop/delta/orders")
table.update("status = 'shipped'", {"status": F.lit("delivered")})      # every shipped order is delivered now
table.delete("status = 'cancelled'")
(table.history()
      .select("version", "operation",
              F.col("operationMetrics")["numRemovedFiles"].alias("files_removed"),
              F.col("operationMetrics")["numAddedFiles"].alias("files_added"))
      .orderBy("version")
      .show())
```

Result:

```
+-------+---------+-------------+-----------+
|version|operation|files_removed|files_added|
+-------+---------+-------------+-----------+
|      0|    WRITE|            0|       NULL|
|      1|   UPDATE|            1|          1|
|      2|   DELETE|            1|          1|
+-------+---------+-------------+-----------+
```

</details>

<details>
<summary><b>A3</b> &nbsp;Append three orders that have an extra column, discount. What happens? Then append them again with ...</summary>

**The first append fails with `[DELTA_METADATA_MISMATCH]`.** A Delta table keeps its schema in the log and rejects writes that don't match it: **schema enforcement**, so a buggy job can't silently pollute the table. When the change is wanted, allow it explicitly with `mergeSchema`: the column is added to the schema, and older rows read it as null (schema evolution).

```python
with_discount = spark.createDataFrame([
    (2001, 1, 101, 1, date(2024, 4, 1), "delivered", 0.1),
    (2002, 2, 105, 2, date(2024, 4, 2), "shipped",   0.0),
    (2003, 4, 106, 1, date(2024, 4, 3), "delivered", 0.2),
], "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING, discount DOUBLE")
try:
    with_discount.write.format("delta").mode("append").save("data/shop/delta/orders")
except Exception as error:
    print("append failed:", str(error).splitlines()[0][:110])
with_discount.write.format("delta").mode("append").option("mergeSchema", "true").save("data/shop/delta/orders")
print(spark.read.format("delta").load("data/shop/delta/orders").columns)
```

Result:

```
append failed: [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
['order_id', 'customer_id', 'product_id', 'quantity', 'order_date', 'status', 'discount']
```

</details>

<details>
<summary><b>A4</b> &nbsp;Read version 0 of the table and the current version. How many rows does each have?</summary>

**16 rows in version 0, 17 now** (2 cancelled orders deleted, 3 orders appended). **Time travel** reads an older version by replaying the log only up to it; its files are still on disk. You can also travel by time (`timestampAsOf`), and in SQL: `SELECT * FROM delta.`/path` VERSION AS OF 0`.

```python
version_0 = spark.read.format("delta").option("versionAsOf", 0).load("data/shop/delta/orders")
print(version_0.count(), "rows in version 0,", spark.read.format("delta").load("data/shop/delta/orders").count(), "now")
```

Result:

```
16 rows in version 0, 17 now
```

</details>

<details>
<summary><b>A5</b> &nbsp;Upsert changes into the table with MERGE: update the orders that exist (only if something changed) and insert ...</summary>

**1 updated and 1 inserted** (order 1002 didn't change, so the condition skipped it). `MERGE INTO`, an **upsert**, matches source rows to target rows on a key and, in one atomic commit, updates the matches, inserts the rest, and can also delete. Running the same MERGE again changes nothing, which makes it the standard way to load changed data. Like UPDATE, it rewrites only the files with matched rows. Naming the columns (`set={...}`, `values={...}`) instead of `UpdateAll`/`InsertAll` keeps the MERGE working when the table has columns the source doesn't (here `discount`, which stays null for the new order).

```python
table = DeltaTable.forPath(spark, "data/shop/delta/orders")       # load it again: the schema may have changed
changes = spark.createDataFrame([
    (1001, 1, 101, 2, date(2024, 1, 10), "delivered"),    # quantity changed from 1 to 2
    (1002, 1, 105, 3, date(2024, 1, 10), "delivered"),    # nothing changed
    (1017, 7, 107, 1, date(2024, 3, 30), "shipped"),      # a new order
], orders.schema)
(table.alias("t")
      .merge(changes.alias("s"), "t.order_id = s.order_id")
      .whenMatchedUpdate(condition="t.quantity <> s.quantity OR t.status <> s.status",
                         set={"quantity": "s.quantity", "status": "s.status"})
      .whenNotMatchedInsert(values={column: f"s.{column}" for column in changes.columns})
      .execute())
(table.history(1)
      .select(F.col("operationMetrics")["numTargetRowsUpdated"].alias("updated"),
              F.col("operationMetrics")["numTargetRowsInserted"].alias("inserted"))
      .show())
```

Result:

```
+-------+--------+
|updated|inserted|
+-------+--------+
|      1|       1|
+-------+--------+
```

</details>

<details>
<summary><b>A6</b> &nbsp;Append 30 small batches to a new table (30 small files). How many files does a filter on one customer read? ...</summary>

**Almost all 30 files before; 1 file after.** `OPTIMIZE` compacts many small files into fewer big ones (here the target size is 512 KB, so you get a few files; the default is 1 GB): it fixes the small file problem. `ZORDER BY customer_id` also orders the rows while rewriting them, so each file holds a narrow range of customer ids. Delta keeps the min and max of every column per file in its log and skips the files that can't contain customer 42: **data skipping**. Z-order can cluster by several columns at once, which plain sorting can't. Both are normal commits: the old small files are only marked as removed.

```python
shutil.rmtree("data/shop/delta/many", ignore_errors=True)
for i in range(30):                                                  # 30 small appends = 30 small files
    (spark.range(i * 10_000, (i + 1) * 10_000)
          .select(F.col("id").alias("order_id"),
                  (F.pmod(F.xxhash64("id"), F.lit(5000)) + 1).alias("customer_id"),
                  F.pmod(F.xxhash64("id", F.lit(2)), F.lit(1000)).cast("double").alias("amount"))
          .coalesce(1).write.format("delta").mode("append").save("data/shop/delta/many"))
many = DeltaTable.forPath(spark, "data/shop/delta/many")

def files_read(customer_id):
    """How many data files a filter on one customer reads."""
    return (spark.read.format("delta").load("data/shop/delta/many")
                 .filter(F.col("customer_id") == customer_id)
                 .select("_metadata.file_path").distinct().count())

print("files:", many.detail().first()["numFiles"], "| read for customer 42:", files_read(42))
spark.conf.set("spark.databricks.delta.optimize.maxFileSize", 512 * 1024)   # 512 KB, so you get several files
many.optimize().executeZOrderBy("customer_id")
print("files:", many.detail().first()["numFiles"], "| read for customer 42:", files_read(42))
```

Result:

```
files: 30 | read for customer 42: 26
files: 5 | read for customer 42: 1
```

</details>

<details>
<summary><b>A7</b> &nbsp;Run VACUUM with 0 hours of retention (the safety check is turned off for this practice table), then try to ...</summary>

**The old files are deleted from disk, and version 0 can no longer be read.** After UPDATE, DELETE, MERGE and OPTIMIZE the replaced files stay on disk, so that time travel and running queries keep working. **VACUUM** deletes the files that no version within the retention period needs (7 days by default), to save storage. Never vacuum with less retention than your longest query or the time travel you need: that's why Delta refuses retention under 7 days unless you turn the check off.

```python
import glob
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")   # allow 0 hours, for practice only
print(len(glob.glob("data/shop/delta/many/*.parquet")), "Parquet files on disk before")
many.vacuum(0)
print(len(glob.glob("data/shop/delta/many/*.parquet")), "after")
import re
try:   # sum, not count: Delta answers count() from the statistics in its log, without reading files
    spark.read.format("delta").option("versionAsOf", 0).load("data/shop/delta/many").agg(F.sum("amount")).show()
except Exception as error:
    print("reading version 0 failed:", re.search(r"\[[A-Z_.]+\]", str(error)).group())
```

Result:

```
35 Parquet files on disk before
Deleted 30 files and directories in a total of 1 directories.
5 after
reading version 0 failed: [FAILED_READ_FILE.FILE_NOT_EXIST]
```

</details>

<details>
<summary><b>A8</b> &nbsp;Create a table with the change data feed turned on, update one order and delete another (cell below). What ...</summary>

**Three rows: order 1001 as `update_preimage` (the old values) and `update_postimage` (the new values), and order 1006 as `delete`, each with its `_commit_version`.** With the **Change Data Feed** on, Delta also records the row-level changes of every commit, so a downstream job can process only what changed since the version it last read (incremental loading, as in Lab 10) instead of reading the whole table again.

```python
orders.write.format("delta").mode("overwrite").option("delta.enableChangeDataFeed", "true").save("data/shop/delta/cdf")
cdf_table = DeltaTable.forPath(spark, "data/shop/delta/cdf")
cdf_table.update("order_id = 1001", {"status": F.lit("returned")})
cdf_table.delete("order_id = 1006")
(spark.read.format("delta").option("readChangeFeed", "true").option("startingVersion", 1).load("data/shop/delta/cdf")
      .select("order_id", "status", "_change_type", "_commit_version")
      .orderBy("_commit_version", "order_id", "_change_type")
      .show())
```

Result:

```
+--------+---------+----------------+---------------+
|order_id|   status|    _change_type|_commit_version|
+--------+---------+----------------+---------------+
|    1001| returned|update_postimage|              1|
|    1001|delivered| update_preimage|              1|
|    1006|  shipped|          delete|              2|
+--------+---------+----------------+---------------+
```

</details>

<details>
<summary><b>B1</b> &nbsp;How many orders and how many items (sum of quantity) are there for each status? Columns: status, orders, ...</summary>

```python
spark.sql("""
    SELECT status, COUNT(*) AS orders, SUM(quantity) AS items
    FROM orders
    GROUP BY status
    ORDER BY status
""").show()
```

</details>

<details>
<summary><b>B2</b> &nbsp;Give every order a size with CASE: small (quantity 1), medium (2 to 3) or large (4 or more), and count the ...</summary>

```python
spark.sql("""
    SELECT CASE WHEN quantity = 1 THEN 'small'
                WHEN quantity <= 3 THEN 'medium'
                ELSE 'large' END AS size,
           COUNT(*) AS orders
    FROM orders
    GROUP BY 1
    ORDER BY size
""").show()
```

</details>

<details>
<summary><b>B3</b> &nbsp;Which customers spent more than 100 on delivered orders (quantity × price)? Columns: name, spent; highest ...</summary>

```python
spark.sql("""
    SELECT c.name, SUM(o.quantity * p.price) AS spent
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN products p ON o.product_id = p.product_id
    WHERE o.status = 'delivered'
    GROUP BY c.name
    HAVING SUM(o.quantity * p.price) > 100
    ORDER BY spent DESC
""").show()
```

</details>

<details>
<summary><b>B4</b> &nbsp;With a CTE and a window function: the best-selling product (most items delivered) of each category. Columns: ...</summary>

```python
spark.sql("""
    WITH items AS (
        SELECT p.category, p.product_name, SUM(o.quantity) AS items
        FROM orders o JOIN products p ON o.product_id = p.product_id
        WHERE o.status = 'delivered'
        GROUP BY p.category, p.product_name
    ), ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY category ORDER BY items DESC, product_name) AS rn
        FROM items
    )
    SELECT category, product_name, items FROM ranked WHERE rn = 1 ORDER BY category
""").show()
```

</details>

<details>
<summary><b>B5</b> &nbsp;For customers 1 and 2, show each order with the customer's running total of items by date. Columns: ...</summary>

```python
spark.sql("""
    SELECT customer_id, order_id, order_date, quantity,
           SUM(quantity) OVER (PARTITION BY customer_id ORDER BY order_date, order_id
                               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS items_so_far
    FROM orders
    WHERE customer_id IN (1, 2)
    ORDER BY customer_id, order_date, order_id
""").show()
```

</details>

<details>
<summary><b>B6</b> &nbsp;Which customers have no orders? Use NOT EXISTS. Columns: customer_id, name.</summary>

```python
spark.sql("""
    SELECT c.customer_id, c.name
    FROM customers c
    WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
""").show()
```

</details>

<details>
<summary><b>B7</b> &nbsp;Delivered revenue per month and its share of the total, in percent (1 decimal). Columns: month, revenue, ...</summary>

```python
spark.sql("""
    WITH monthly AS (
        SELECT MONTH(o.order_date) AS month, SUM(o.quantity * p.price) AS revenue
        FROM orders o JOIN products p ON o.product_id = p.product_id
        WHERE o.status = 'delivered'
        GROUP BY MONTH(o.order_date)
    )
    SELECT month, revenue, ROUND(revenue / SUM(revenue) OVER () * 100, 1) AS share_pct
    FROM monthly
    ORDER BY month
""").show()
```

</details>

<details>
<summary><b>B8</b> &nbsp;For each customer, the first and the last order date and the days between them. Columns: name, first_order, ...</summary>

```python
spark.sql("""
    SELECT c.name, MIN(o.order_date) AS first_order, MAX(o.order_date) AS last_order,
           DATEDIFF(MAX(o.order_date), MIN(o.order_date)) AS days_between
    FROM orders o JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.name
    ORDER BY c.name
""").show()
```

</details>

<details>
<summary><b>B9</b> &nbsp;Create the Delta table shop_orders from orders with CREATE TABLE ... AS SELECT, mark every shipped order as ...</summary>

```python
spark.sql("DROP TABLE IF EXISTS shop_orders")
spark.sql("CREATE TABLE shop_orders USING DELTA AS SELECT * FROM orders")
spark.sql("UPDATE shop_orders SET status = 'delivered' WHERE status = 'shipped'")
spark.sql("SELECT status, COUNT(*) AS orders FROM shop_orders GROUP BY status ORDER BY status").show()
```

</details>

<details>
<summary><b>B10</b> &nbsp;MERGE the view order_updates into shop_orders: update the matching orders, insert the new ones. Then show ...</summary>

```python
spark.sql("""
    MERGE INTO shop_orders t
    USING order_updates s
    ON t.order_id = s.order_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
spark.sql("SELECT order_id, quantity, status FROM shop_orders WHERE order_id IN (1001, 1017) ORDER BY order_id").show()
```

</details>

<details>
<summary><b>B11</b> &nbsp;Count the rows of shop_orders in versions 0, 1 and 2 with VERSION AS OF. Columns: version, rows.</summary>

```python
spark.sql("""
    SELECT 0 AS version, COUNT(*) AS rows FROM shop_orders VERSION AS OF 0
    UNION ALL SELECT 1, COUNT(*) FROM shop_orders VERSION AS OF 1
    UNION ALL SELECT 2, COUNT(*) FROM shop_orders VERSION AS OF 2
""").show()
```

</details>